<a href="https://colab.research.google.com/github/RezeneG/NLP_Final_2415644/blob/main/NLP_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()   # select your CSV file

Saving Consumer Complaints Dataset for NLP.csv to Consumer Complaints Dataset for NLP.csv


In [ ]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')

# Load the CSV (use the exact filename from upload)
df = pd.read_csv('Consumer Complaints Dataset for NLP.csv')
print("Columns:", df.columns.tolist())
print("Shape:", df.shape)

# Drop unnecessary index column if present
if 'Unnamed: 0' in df.columns:
    df = df.drop('Unnamed: 0', axis=1)

# Drop missing narratives
df = df.dropna(subset=['narrative'])

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


Columns: ['Unnamed: 0', 'product', 'narrative']
Shape: (162421, 3)


In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(w) for w in tokens if w not in stop_words]
    return ' '.join(tokens)

df['clean_narrative'] = df['narrative'].apply(clean_text)
df = df[df['clean_narrative'].str.len() > 10]
print(f"Rows after cleaning: {len(df)}")

Rows after cleaning: 162244


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['label'] = le.fit_transform(df['product'])
print("Class mapping:", dict(zip(le.classes_, le.transform(le.classes_))))

X = df['clean_narrative']
y = df['label']

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

Class mapping: {'credit_card': np.int64(0), 'credit_reporting': np.int64(1), 'debt_collection': np.int64(2), 'mortgages_and_loans': np.int64(3), 'retail_banking': np.int64(4)}
Train: 113570, Val: 24337, Test: 24337


In [ ]:
# Training subset: 8,000 samples
X_train_sub, _, y_train_sub, _ = train_test_split(
    X_train, y_train,
    train_size=8000,
    stratify=y_train,
    random_state=42
)
print(f"Training subset: {len(X_train_sub)}")

# Validation subset: 2,000 samples
X_val_sub, _, y_val_sub, _ = train_test_split(
    X_val, y_val,
    train_size=2000,
    stratify=y_val,
    random_state=42
)
print(f"Validation subset: {len(X_val_sub)}")

Training subset: 8000
Validation subset: 2000


In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

GPU available: True
GPU name: Tesla T4


In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("Still on CPU – please repeat the steps.")

GPU available: True
GPU name: Tesla T4


In [ ]:
import torch; print(torch.cuda.is_available())

False


In [ ]:
!pip install transformers datasets accelerate -q

In [ ]:
import torch
import time
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

def prepare_dataset(texts, labels, tokenizer, max_length=64):
    ds = Dataset.from_dict({'text': texts, 'label': labels})
    def tokenize_func(examples):
        return tokenizer(examples['text'], padding='max_length',
                         truncation=True, max_length=max_length)
    ds = ds.map(tokenize_func, batched=True, remove_columns=['text'])
    ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
    return ds

def train_transformer(model_name, train_texts, train_labels, val_texts, val_labels,
                      learning_rate=2e-5, batch_size=2, epochs=3, max_length=64,
                      gradient_accumulation_steps=4, fp16=True):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=5)

    # Enable gradient checkpointing only if supported
    if hasattr(model, 'gradient_checkpointing_enable'):
        try:
            model.gradient_checkpointing_enable()
        except ValueError:
            print(f"Gradient checkpointing not supported for {model_name}, continuing without it.")

    train_ds = prepare_dataset(train_texts, train_labels, tokenizer, max_length)
    val_ds   = prepare_dataset(val_texts, val_labels, tokenizer, max_length)

    output_dir = f'./{model_name.replace("/", "_")}_{int(time.time())}'

    training_args = TrainingArguments(
        output_dir=output_dir,
        eval_strategy='epoch',
        save_strategy='epoch',
        save_total_limit=1,
        learning_rate=learning_rate,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=epochs,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model='accuracy',
        logging_dir='./logs',
        gradient_accumulation_steps=gradient_accumulation_steps,
        fp16=fp16,
        dataloader_pin_memory=False,
    )

    def compute_metrics(p):
        preds = p.predictions.argmax(-1)
        return {'accuracy': (preds == p.label_ids).mean()}

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics,
    )
    trainer.train()
    return trainer, tokenizer

In [ ]:
torch.cuda.empty_cache()
trainer_dbert1, tokenizer_dbert1 = train_transformer(
    'distilbert-base-uncased',
    X_train_sub.tolist(), y_train_sub,
    X_val_sub.tolist(), y_val_sub,
    learning_rate=2e-5, batch_size=2, epochs=3, max_length=64,
    gradient_accumulation_steps=4, fp16=True
)

torch.cuda.empty_cache()
trainer_dbert2, tokenizer_dbert2 = train_transformer(
    'distilbert-base-uncased',
    X_train_sub.tolist(), y_train_sub,
    X_val_sub.tolist(), y_val_sub,
    learning_rate=5e-5, batch_size=2, epochs=4, max_length=64,
    gradient_accumulation_steps=4, fp16=True
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


In [ ]:
torch.cuda.empty_cache()
trainer_albert1, tokenizer_albert1 = train_transformer(
    'albert-base-v2',
    X_train_sub.tolist(), y_train_sub,
    X_val_sub.tolist(), y_val_sub,
    learning_rate=2e-5, batch_size=2, epochs=3, max_length=64,
    gradient_accumulation_steps=4, fp16=True
)

torch.cuda.empty_cache()
trainer_albert2, tokenizer_albert2 = train_transformer(
    'albert-base-v2',
    X_train_sub.tolist(), y_train_sub,
    X_val_sub.tolist(), y_val_sub,
    learning_rate=5e-5, batch_size=2, epochs=4, max_length=64,
    gradient_accumulation_steps=4, fp16=True
)

config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/760k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/47.4M [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['albert.embeddings.LayerNorm.weight', 'albert.embeddings.LayerNorm.bias', 'albert.encoder.albert_layer_groups.0.albert_layers.0.attention.LayerNorm.weight', 'albert.encoder.albert_layer_groups.0.albert_layers.0.attention.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['albert.embeddings.LayerNorm.beta', 'albert.embeddings.LayerNorm.gamma', 'albert.encoder.albert_layer_groups.0.albert_layers.0.attention.LayerNorm.beta', 'albert.encoder.albert_layer_groups.0.albert_layers.0.attention.LayerNorm.gamma'].


In [ ]:
torch.cuda.empty_cache()
trainer_deberta1, tokenizer_deberta1 = train_transformer(
    'microsoft/deberta-v3-small',
    X_train_sub.tolist(), y_train_sub,
    X_val_sub.tolist(), y_val_sub,
    learning_rate=2e-5, batch_size=2, epochs=3, max_length=64,
    gradient_accumulation_steps=4, fp16=False
)

torch.cuda.empty_cache()
trainer_deberta2, tokenizer_deberta2 = train_transformer(
    'microsoft/deberta-v3-small',
    X_train_sub.tolist(), y_train_sub,
    X_val_sub.tolist(), y_val_sub,
    learning_rate=5e-5, batch_size=2, epochs=4, max_length=64,
    gradient_accumulation_steps=4, fp16=False
)

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/286M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['deberta.embeddings.LayerNorm.weight', 'deberta.embeddings.LayerNorm.bias', 'deberta.encoder.layer.0.attention.output.LayerNorm.weight', 'deberta.encoder.layer.0.attention.output.LayerNorm.bias', 'deberta.encoder.layer.0.output.LayerNorm.weight', 'deberta.encoder.layer.0.output.LayerNorm.bias', 'deberta.encoder.layer.1.attention.output.LayerNorm.weight', 'deberta.encoder.layer.1.attention.output.LayerNorm.bias', 'deberta.encoder.layer.1.output.LayerNorm.weight', 'deberta.encoder.layer.1.output.LayerNorm.bias', 'deberta.encoder.layer.2.attention.output.LayerNorm.weight', 'deberta.encoder.layer.2.attention.output.LayerNorm.bias', 'deberta.encoder.layer.2.output.LayerNorm.weight', 'deberta.encoder.layer.2.output.LayerNorm.bias', 'deberta.encoder.layer.3.attention.output.LayerNorm.weight', 'deberta.encoder.layer.3.attention.output.LayerNorm.bias', 'deberta.encoder.layer.3.output.LayerNorm.weight', 'deberta.encoder.layer.3.output.Laye

In [ ]:
from sklearn.metrics import classification_report

def evaluate_transformer(trainer, tokenizer, test_texts, test_labels, name):
    test_ds = prepare_dataset(test_texts, test_labels, tokenizer, max_length=128)
    predictions = trainer.predict(test_ds)
    y_pred = np.argmax(predictions.predictions, axis=1)
    print(f"\n{name} - Test Performance (full test set):")
    print(classification_report(test_labels, y_pred, target_names=le.classes_))
    return y_pred

evaluate_transformer(trainer_dbert1, tokenizer_dbert1, X_test.tolist(), y_test, "DistilBERT Config1")
evaluate_transformer(trainer_dbert2, tokenizer_dbert2, X_test.tolist(), y_test, "DistilBERT Config2")
evaluate_transformer(trainer_albert1, tokenizer_albert1, X_test.tolist(), y_test, "ALBERT Config1")
evaluate_transformer(trainer_albert2, tokenizer_albert2, X_test.tolist(), y_test, "ALBERT Config2")
evaluate_transformer(trainer_deberta1, tokenizer_deberta1, X_test.tolist(), y_test, "DeBERTa Config1")
evaluate_transformer(trainer_deberta2, tokenizer_deberta2, X_test.tolist(), y_test, "DeBERTa Config2")

Map:   0%|          | 0/24337 [00:00<?, ? examples/s]


DistilBERT Config1 - Test Performance (full test set):
                     precision    recall  f1-score   support

        credit_card       0.78      0.74      0.76      2335
   credit_reporting       0.90      0.93      0.91     13659
    debt_collection       0.76      0.71      0.74      3467
mortgages_and_loans       0.84      0.81      0.83      2848
     retail_banking       0.81      0.85      0.83      2028

           accuracy                           0.86     24337
          macro avg       0.82      0.81      0.81     24337
       weighted avg       0.86      0.86      0.86     24337



Map:   0%|          | 0/24337 [00:00<?, ? examples/s]


DistilBERT Config2 - Test Performance (full test set):
                     precision    recall  f1-score   support

        credit_card       0.78      0.73      0.76      2335
   credit_reporting       0.90      0.93      0.91     13659
    debt_collection       0.75      0.72      0.74      3467
mortgages_and_loans       0.85      0.79      0.81      2848
     retail_banking       0.83      0.85      0.84      2028

           accuracy                           0.86     24337
          macro avg       0.82      0.80      0.81     24337
       weighted avg       0.85      0.86      0.85     24337



Map:   0%|          | 0/24337 [00:00<?, ? examples/s]

...

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Map:   0%|          | 0/24337 [00:00<?, ? examples/s]


DeBERTa Config2 - Test Performance (full test set):
                     precision    recall  f1-score   support

        credit_card       0.10      1.00      0.18      2335
   credit_reporting       0.00      0.00      0.00     13659
    debt_collection       0.00      0.00      0.00      3467
mortgages_and_loans       0.00      0.00      0.00      2848
     retail_banking       0.00      0.00      0.00      2028

           accuracy                           0.10     24337
          macro avg       0.02      0.20      0.04     24337
       weighted avg       0.01      0.10      0.02     24337



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


array([0, 0, 0, ..., 0, 0, 0])